# Beltrami Flow Model


## Setup


In [31]:
import sys
from dataclasses import asdict, replace
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
sys.path.append(str(ROOT / "src"))

import torch

from models import (
    MLP, MLPConfig
)
from train import (
    TrainConfig, OptimizerConfig, run_experiment
)
from problems.beltrami import (
    BeltramiProblem, BeltramiConfig, Domain3DTime
)
from utils import (
    get_device, make_experiment_dir, make_run_dir, make_sweep_dir,
    rebuild_sweep_summary_csv, save_manifest
)


## Problem Factory


In [32]:
def make_problem(
    Re: float = 1.0,
    lambda_bc: float = 1.0,
    lambda_ic: float = 1.0,
    seed: int = 0,
):
    return BeltramiProblem(
        domain=Domain3DTime(),
        cfg=BeltramiConfig(
            Re=Re,
            lambda_bc=lambda_bc,
            lambda_ic=lambda_ic,
            include_final_time_bc=True,
            lambda_final=1.0
        ),
        seed=seed,
    )


## Base Config


In [ ]:
seeds = [5246, 16567, 19943, 29393, 30373]   # list of seeds (even if only one seed, we need a list)
mlp_layer_dims = (4, 50, 50, 50, 50, 4)
device_preference = "cuda"  # cpu, cuda, or auto
dtype = torch.float64
results_root = ROOT / "runs"
sweep_name = "v1_rtx4090"


In [34]:
device = get_device(device_preference)
problem_name = BeltramiProblem.name
experiment_index = {}

# Define neural network
model_cfg = MLPConfig(
    layer_dims=mlp_layer_dims,
    activation="tanh",
    init="xavier_uniform",  # also known as Glorot uniform
)
num_params = sum(p.numel() for p in MLP(model_cfg).parameters())

# Instantiate basic Trainer configuration
#  - seed is injected later
#  - can be changed later using 'replace'
#  - for Beltrami, nx_* * ny_* is interpreted as the total number of
#    fixed Sobol spatio-temporal interior / validation points
train_cfg = TrainConfig(
    steps=100_000,            # max. steps within time constraint
    time_constraint_s=12.*60.0,
    log_every=100,
    dtype=dtype,
    nx_int=51, ny_int=51,    # interior samples in (x, y, z, t)
    n_bnd=26*26,             # boundary, and initial- and final conditon samples
    nx_val=100, ny_val=100,  # validation samples in (x, y, z, t)
    resample_train_batch_each_step=False,
)

print("MLP layers:", model_cfg.layer_dims)
print("parameters:", num_params)
print("problem:", problem_name)
print("device:", device)
print("dtype:", dtype)
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("cuda:", torch.cuda.get_device_name(0))


MLP layers: (4, 50, 50, 50, 50, 4)
parameters: 8104
problem: beltrami
device: cuda
dtype: torch.float64
torch: 2.11.0+cu128
cuda: NVIDIA GeForce RTX 5060 Laptop GPU


## Optimizer Configs


In [35]:
adam_opt_cfg = OptimizerConfig(
    name="adam",
    lr=1e-3,
    cosine_annealing=True,  # until train_cgf.steps
)

lbfgs_opt_cfg = OptimizerConfig(
    name="lbfgs",
    lr=1.0,
    kwargs=dict(
        max_iter=20,
        history_size=100,
        line_search_fn="strong_wolfe",
        tolerance_grad=1e-7,
        tolerance_change=1e-9,
    ),
)

gnng_dense_opt_cfg = OptimizerConfig(
    name="gauss_newton",
    lr=1.0,
    kwargs=dict(
        backend="dense",
        damping_cap=1e-3,
        damping_floor=1e-7,
        damping_scale=6e-5,
        do_line_search=True,
        line_search_steps=31,
    ),
)

gnng_matfree_opt_cfg = OptimizerConfig(
    name="gauss_newton",
    lr=1.0,
    kwargs=dict(
        backend="matfree",
        use_cg_precond=True,
        precond_nx_int=20,
        precond_ny_int=20,
        precond_n_bnd=256,
        damping_cap=1e-4,
        damping_floor=1e-6,
        damping_scale=3e-5,
        cg_rtol=1e-3,
        cg_atol=1e-5,
        cg_maxiter=200,
        do_line_search=True,
        line_search_steps=31,
    ),
)

## Execution
Sweep over the defined experiments and seeds.

In [ ]:
experiment_specs = [
    dict(
        experiment_name="adam", 
        opt_cfg=adam_opt_cfg,
        train_cfg=replace(train_cfg, log_every=25),
    ),
    dict(
        experiment_name="lbfgs", 
        opt_cfg=lbfgs_opt_cfg, 
        train_cfg=replace(train_cfg, log_every=3)
    ),
    dict(
        experiment_name="gnng_dense_v0",
        opt_cfg=gnng_dense_opt_cfg, 
        train_cfg=replace(train_cfg, log_every=1)
    ),
    dict(
        experiment_name="gnng_dense_v1",
        opt_cfg=replace(
            gnng_dense_opt_cfg, 
            kwargs=dict(
                gnng_dense_opt_cfg.kwargs,
                damping_cap=1e-5,
                damping_floor=1e-16,
                damping_scale=1.0,
            ),
        ), 
        train_cfg=replace(train_cfg, log_every=1)
    ),
    dict(
        experiment_name="gnng_dense_v2",
        opt_cfg=gnng_dense_opt_cfg, 
        train_cfg=replace(train_cfg, nx_int=100, ny_int=100, n_bnd=31*31, log_every=1)
    ),
    dict(
        experiment_name="gnng_matfree_v0", 
        opt_cfg=gnng_matfree_opt_cfg, 
        train_cfg=replace(train_cfg, log_every=2)
    ),
    dict(
        experiment_name="gnng_matfree_v1", 
        opt_cfg=replace(
            gnng_matfree_opt_cfg, 
            kwargs=dict(
                gnng_matfree_opt_cfg.kwargs,
                damping_cap=1e-5,
                damping_floor=1e-16,
                damping_scale=1.0,
                cg_atol=1e-3,
                cg_rtol=1e-5,
            ),
        ), 
        train_cfg=replace(train_cfg, log_every=2)
    ),
    dict(
        experiment_name="gnng_matfree_v2", 
        opt_cfg=gnng_matfree_opt_cfg, 
        train_cfg=replace(
            train_cfg, nx_int=100, ny_int=100, n_bnd=31*31, log_every=2
        )
    ),
]

selected_experiment_names = [
    "adam",
    "lbfgs",
    "gnng_dense_v0",
    "gnng_dense_v1",
    "gnng_dense_v2",
    "gnng_matfree_v0",
    "gnng_matfree_v1",
    "gnng_matfree_v2",
]

specs_by_name = {spec["experiment_name"]: spec for spec in experiment_specs}

missing = [name for name in selected_experiment_names if name not in specs_by_name]
if missing:
    raise ValueError(f"Unknown experiment name(s): {missing}")

selected_specs = [specs_by_name[name] for name in selected_experiment_names]

sweep_dir = make_sweep_dir(results_root, problem_name, sweep_name=sweep_name)
save_manifest(
    sweep_dir,
    problem_name=problem_name,
    seeds=seeds,
    experiment_names=selected_experiment_names,
    shared_config={
        "model": asdict(model_cfg),
        "train": asdict(train_cfg),
        "dtype": str(dtype),
        "device": str(device),
    },
)

for spec in selected_specs:
    experiment_dir = make_experiment_dir(sweep_dir, spec["experiment_name"])

    for seed in seeds:
        run_dir = make_run_dir(experiment_dir, seed)
        problem = make_problem(seed=seed)

        history, summary_payload, save_path = run_experiment(
            experiment_name=spec["experiment_name"],
            problem=problem,
            opt_cfg=spec["opt_cfg"],
            train_cfg=replace(spec["train_cfg"], seed=seed),
            model_cfg=model_cfg,
            seed=seed,
            dtype=dtype,
            device=device,
            sweep_name=sweep_name,
            repo_root=ROOT,
            run_dir=run_dir,
            verbose=True,
        )

        rebuild_sweep_summary_csv(sweep_dir)


{'step': '000001', 'experiment': 'gnng_dense_v0', 'wall time [s]': '    29.6', 'loss': '1.77e+01', 'rel L2 (mean)': '9.79e-01', 'backend': 'dense', 'damping': '1.00e-03', 'step size': '3.41e-02', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '6613.0'}
{'step': '000002', 'experiment': 'gnng_dense_v0', 'wall time [s]': '    59.2', 'loss': '1.71e+01', 'rel L2 (mean)': '9.65e-01', 'backend': 'dense', 'damping': '1.00e-03', 'step size': '3.41e-02', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '6613.0'}
{'step': '000003', 'experiment': 'gnng_dense_v0', 'wall time [s]': '    88.6', 'loss': '1.17e+01', 'rel L2 (mean)': '8.18e-01', 'backend': 'dense', 'damping': '1.00e-03', 'step size': '1.58e-01', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '6613.0'}
{'step': '000004', 'experiment': 'gnng_dense_v0', 'wall time [s]': '   118.1', 'loss': '7.31e+00', 'rel L2 (mean)': '6.83e-01', 'backend': 'dense', 'damping': '7.01e-04', 'step size': '2.15e-01', 'lr': '1.00e+00', 'gpu_peak_mem_mb': '6613.0'}
{'step': '000005', 'experime